# Evaluating trained models on Kaggle

Scores the checkpoints written by `05_kaggle_training.ipynb` with `mammo-seg
evaluate`: each fold's model is run on the images it did not train on (its own
validation fold) under a grid of thresholds and minimum region sizes, next to the
baseline of predicting nothing. Research use only: this is not a medical device.

**Once per notebook (settings panel):**
- *Add input*: the INbreast dataset **and the output of the training notebook** (a
  saved version of it; its `runs` folder holds the checkpoints).
- Accelerator: *GPU T4* (or P100), chosen before running anything. Evaluating is much
  faster on a GPU, though it also works without one if `--device cpu` is added to
  `EVALUATE_OPTIONS`.
- Internet: *on* (the repository is downloaded). No pretrained weights are needed.

**Run** the cells in order; only the first one needs editing. Never put `kaggle.json`
or any other credential in a notebook.

**The test set** is scored by adding `--test` to `EVALUATE_OPTIONS`. Do that once, at
the very end, with the threshold and minimum region size you chose on the validation
folds: choosing a setting on the test set makes the final number optimistic.

In [ ]:
from pathlib import Path

FOLDS = [0, 1, 2, 3, 4]
# More options of `mammo-seg evaluate`, for example "--checkpoint-name best.pt" or
# "--thresholds 0.4 0.5 --min-areas-mm2 0 5". "--test" scores the held-out test set.
EVALUATE_OPTIONS = ""
REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
REPO_REF = "main"  # branch or tag to install

## 1. Setup

Installs the package, finds the dataset and the checkpoints, and rebuilds the derived
data (manifest, splits, masks, preprocessed images: a few minutes). The splits are
made with the same seed as in training, so each fold has the same validation images;
a checkpoint trained for another fold is refused.

In [ ]:
WORK = Path("/kaggle/working")
REPO = WORK / "repo"
PROCESSED = WORK / "data" / "processed"
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
MASKS = PROCESSED / "masks"
PREPROCESSED = PROCESSED / "preprocessed"
test_set = "--test" in EVALUATE_OPTIONS
OUTPUT = WORK / f"evaluation_{'test' if test_set else 'validation'}.json"

found = sorted(Path("/kaggle/input").rglob("INbreast.csv"))
assert found, "attach the INbreast dataset as an input of the notebook"
DATASET_DIR = found[0].parent
print("dataset:", DATASET_DIR)

checkpoints = sorted(Path("/kaggle/input").rglob("fold_0/last.pt"))
assert checkpoints, "attach the output of the training notebook as an input"
RUNS = checkpoints[0].parent.parent
print("runs:", RUNS)

!test -d {REPO} || git clone -q --branch {REPO_REF} {REPO_URL} {REPO}
!pip install -q "{REPO}[cloud]"
!nvidia-smi -L

PROCESSED.mkdir(parents=True, exist_ok=True)
if not MANIFEST.exists():
    !mammo-seg manifest --dataset-dir "{DATASET_DIR}" --output {MANIFEST}
if not SPLITS.exists():
    !mammo-seg splits --folds 5 --test-fraction 0.2 \
        --manifest {MANIFEST} --output {SPLITS}
if not MASKS.exists():
    !mammo-seg masks --manifest {MANIFEST} --output-dir {MASKS}
if not (PREPROCESSED / "preprocessing.json").exists():
    !mammo-seg preprocess --manifest {MANIFEST} --dataset-dir "{DATASET_DIR}" \
        --masks-dir {MASKS} --output-dir {PREPROCESSED}

## 2. Evaluate

The table pools the folds: every image counts once, scored by the model that did not
train on it. Columns: Dice, sensitivity and precision on the images with a mass; the
share of images without a mass with any prediction left (`false alarms`) and the
mean number of predicted pixels on them (`FP pixels`); the Dice over all images; and
the *balanced* Dice, the mean of the Dice on images with a mass and on images
without, which weighs both kinds equally. `*` marks the best balanced Dice, and the
rows below the table show every fold at that setting. A model that predicts nothing
scores the share of images without a mass on the all-images Dice, so the all-images
Dice means little unless it is above the `predict nothing` row. Everything is also
written to `evaluation_validation.json` (or `_test`) in the output.

In [ ]:
folds = " ".join(str(fold) for fold in FOLDS)
!mammo-seg evaluate --manifest {MANIFEST} --splits {SPLITS} \
    --preprocessed-dir {PREPROCESSED} --runs-dir {RUNS} --output {OUTPUT} \
    --device cuda --num-workers 2 {EVALUATE_OPTIONS} --folds {folds}